# Lumbar Discharge Disposition - XGBoost Inference with SHAP

**Inference pipeline for predicting non-home discharge after lumbar fusion surgery.**

## Overview
This notebook uses a pre-trained XGBoost model to predict discharge disposition (home vs. non-home facility) based on 10 LASSO-selected clinical features.

**Input**: CSV file with raw clinical features (automatic preprocessing included)  
**Output**: 
- Performance metrics (AUC-ROC, Accuracy, Precision, Recall, F1, Confusion Matrix)
- ROC curve visualization
- SHAP beeswarm plot (global feature importance)
- SHAP force plots for top 10 samples (local explanations, requires ground truth)

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import shap
import re

from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

RANDOM_SEED = 55
TARGET = "discharge_disposition"

# Configuration for SHAP force plots
CFG = {
    "N_correct_class_0": 5,       # Top 5 Home (Good)
    "N_correct_class_1": 5,       # Top 5 Non-Home (Bad)
    "n_features_plot": 10,        # Number of top features to show
    "figsize": (20, 5),
    "text_rotation": 30
}

print("✅ Setup complete")

In [ ]:
# 10 features selected by LASSO (frequency >= 3/5 folds) for DISCHARGE task
SELECTED_FEATURES = [
    "cont__age_at_admit",
    "cat__marital_status_1",
    "cat__gender_1",
    "cat__surgical_approach_group_1",
    "cat__surgical_approach_group_2",
    "cat__osteoporosis_1",
    "cat__psych_1",
    "cont__hgb",
    "cont__magnesium",
    "cat__htn_1"
]

# Raw feature columns needed for preprocessing
RAW_NUMERIC_FEATURES = ["age_at_admit", "hgb", "magnesium"]
RAW_CATEGORICAL_FEATURES = ["marital_status", "gender", "surgical_approach_group", "osteoporosis", "psych", "htn"]

# Readable feature names for SHAP plots
FEATURE_NAMES = {
    "cont__age_at_admit": "Age at Admission",
    "cat__marital_status_1": "Marital Status (Married)",
    "cat__gender_1": "Gender (Male)",
    "cat__surgical_approach_group_1": "Surgical Approach (Anterior)",
    "cat__surgical_approach_group_2": "Surgical Approach (Combined)",
    "cat__osteoporosis_1": "Osteoporosis",
    "cat__psych_1": "Psychiatric Disorder",
    "cont__hgb": "Hemoglobin",
    "cont__magnesium": "Magnesium",
    "cat__htn_1": "Hypertension"
}

print(f"✅ Using {len(SELECTED_FEATURES)} encoded features for DISCHARGE task")
print(f"✅ Raw features: {len(RAW_NUMERIC_FEATURES)} numeric + {len(RAW_CATEGORICAL_FEATURES)} categorical")

## 2. Build Preprocessing Pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Build preprocessing pipeline (same as training)
preprocessor = ColumnTransformer(
    transformers=[
        ("cont", StandardScaler(), RAW_NUMERIC_FEATURES),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), 
         RAW_CATEGORICAL_FEATURES)
    ],
    remainder="drop"
)

print("✅ Preprocessing pipeline created")
print(f"   Numeric features: {RAW_NUMERIC_FEATURES}")
print(f"   Categorical features: {RAW_CATEGORICAL_FEATURES}")

## 3. Load XGBoost Model

Pre-trained model is located at `models/xgb_model.json`.

In [ ]:
# ============================================================================
# CONFIGURE MODEL PATH
# ============================================================================
# Replace with your XGBoost model path
MODEL_PATH = Path("models/xgb_model.json")

# Load XGBoost native booster
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"XGBoost model not found at: {MODEL_PATH.absolute()}")

xgb_booster = xgb.Booster()
xgb_booster.load_model(str(MODEL_PATH))

print(f"✅ Loaded XGBoost model from: {MODEL_PATH.absolute()}")

## 4. Load Input Data (Raw CSV)

**Input Requirements:**
- CSV file with raw clinical features (not encoded)
- Required features: `age_at_admit`, `hgb`, `magnesium`, `marital_status`, `gender`, `surgical_approach_group`, `osteoporosis`, `psych`, `htn`
- Optional: `discharge_disposition` column for evaluation (0=home, 1=non-home)

The notebook will automatically preprocess these raw features.

In [ ]:
# ============================================================================
# CONFIGURE INPUT DATA
# ============================================================================
INPUT_PATH = Path("../Data/sample_synthetic.csv")  # Replace with your input CSV path
HAS_LABELS = True  # Set to True if your CSV contains ground truth labels

# ============================================================================
# LOAD RAW DATA
# ============================================================================
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} samples with {len(df.columns)} columns")

# Force categorical dtypes
for col in RAW_CATEGORICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Validate required raw features
all_raw_features = RAW_NUMERIC_FEATURES + RAW_CATEGORICAL_FEATURES
missing = [f for f in all_raw_features if f not in df.columns]
if missing:
    raise ValueError(f"Missing required raw features: {missing}")

# Load ground truth if available
y_true = None
if HAS_LABELS and TARGET in df.columns:
    y_true = df[TARGET].astype(int).values
    print(f"✓ Ground truth found: {TARGET} (n={len(y_true)})")
else:
    print("⚠ No ground truth (prediction only)")

print(f"✅ Raw data loaded: {df.shape}")
print(f"✅ Required raw features: {all_raw_features}")

## 5. Preprocess Data & Extract Selected Features

Automatically encode raw features and select the 10 LASSO features.

In [ ]:
# Extract raw features
X_raw = df[all_raw_features].copy()

# Fit and transform (fit on inference data since we don't have training stats)
# NOTE: Ideally, you should save the fitted preprocessor from training
X_encoded = preprocessor.fit_transform(X_raw)

# Get encoded feature names
encoded_feature_names = preprocessor.get_feature_names_out().tolist()
print(f"✅ Encoded features: {len(encoded_feature_names)} total")

# Create DataFrame with encoded features
X_encoded_df = pd.DataFrame(X_encoded, columns=encoded_feature_names)

# Select only the 10 LASSO-selected features
missing_features = [f for f in SELECTED_FEATURES if f not in encoded_feature_names]
if missing_features:
    print(f"⚠ Warning: Some selected features not found in encoded data: {missing_features}")
    # Fill missing with 0
    for f in missing_features:
        X_encoded_df[f] = 0

# Extract final feature matrix (10 features)
X = X_encoded_df[SELECTED_FEATURES].values
print(f"✅ Final feature matrix: {X.shape} (10 LASSO-selected features)")

## 6. Predict with XGBoost

Generate probability predictions and binary classifications.

In [ ]:
# Create DMatrix
dmat = xgb.DMatrix(X, feature_names=SELECTED_FEATURES)

# Predict probabilities
y_proba = xgb_booster.predict(dmat)
y_pred = (y_proba >= 0.5).astype(int)

print(f"✅ Predictions generated")
print(f"   Mean probability: {np.mean(y_proba):.3f}")
print(f"   Predicted class distribution: {np.bincount(y_pred)}")

## 7. Compute Metrics

**Note**: Metrics are only computed when ground truth labels are available (`HAS_LABELS=True`).

In [ ]:
if y_true is not None:
    # Compute metrics
    auc = roc_auc_score(y_true, y_proba)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Print results
    print("\n" + "="*80)
    print("📊 XGBOOST PERFORMANCE METRICS")
    print("="*80)
    print(f"AUC-ROC:    {auc:.4f}")
    print(f"Accuracy:   {acc:.4f}")
    print(f"Precision:  {prec:.4f}")
    print(f"Recall:     {rec:.4f}")
    print(f"F1-Score:   {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  TN: {tn:4d}  |  FP: {fp:4d}")
    print(f"  FN: {fn:4d}  |  TP: {tp:4d}")
    print("="*80)
else:
    print("⚠ No ground truth - metrics skipped")

## 8. SHAP Analysis - Beeswarm Plot

**Global feature importance** showing how each feature impacts model predictions across all samples.

The beeswarm plot displays:
- Feature importance (x-axis: SHAP value)
- Feature values (color: red=high, blue=low)
- Distribution across samples (y-axis spread)

In [ ]:
# Compute SHAP values using XGBoost native method
shap_values = xgb_booster.predict(dmat, pred_contribs=True)

# Remove bias term (last column)
shap_values_clean = shap_values[:, :-1]

print(f"✅ SHAP values computed: shape={shap_values_clean.shape}")

# Map feature names to readable names
readable_names = [FEATURE_NAMES.get(f, f) for f in SELECTED_FEATURES]

# Create beeswarm plot
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values_clean,
    X,
    feature_names=readable_names,
    show=False
)
plt.title("XGBoost Feature Importance (SHAP)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_beeswarm_XGB.png", dpi=300, bbox_inches='tight')
print("💾 Saved: shap_beeswarm_XGB.png")
plt.show()

## 9. ROC Curve Visualization

**Note**: ROC curve requires ground truth labels for evaluation.

In [ ]:
if y_true is not None:
    from sklearn.metrics import roc_curve
    
    # Compute ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    
    # Plot ROC curve
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'XGBoost (AUC = {auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve - XGBoost', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("roc_curve_XGB.png", dpi=300, bbox_inches='tight')
    print("💾 Saved: roc_curve_XGB.png")
    plt.show()
else:
    print("⚠ No ground truth - ROC curve skipped")

## 10. SHAP Force Plots

**Local explanations** for individual predictions.

Force plots show how each feature pushes the prediction higher (red) or lower (blue) from the base value. 

**Note**: Requires ground truth labels to identify top correctly predicted samples.

In [ ]:
# =============================================================================
# SHAP FORCE PLOT CONFIGURATION & HELPER FUNCTIONS
# =============================================================================

# --- A. VALUE MAPS: Convert numeric values to readable text ---
VALUE_MAPS = {
    "gender": {0: "Female", 1: "Male"},
    "marital_status": {0: "Single", 1: "Married", 2: "Divorced", 3: "Widowed", 4: "Unknown"},
    "surgical_approach_group": {0: "Posterior", 1: "Anterior", 2: "Combined"},
    "binary": {0: "No", 1: "Yes"}  # For all binary variables
}

BINARY_COLS = ["osteoporosis", "psych", "htn"]

# --- B. PRETTY NAMES: Display names for features ---
PRETTY_NAMES = {
    "age_at_admit": "Age",
    "hgb": "Hemoglobin",
    "magnesium": "Magnesium",
    "gender": "Gender",
    "marital_status": "Marital Status",
    "surgical_approach_group": "Surgical Approach",
    "osteoporosis": "Osteoporosis",
    "psych": "Psychiatric Disorder",
    "htn": "Hypertension"
}

# --- C. ONE-HOT GROUPS: Define how to aggregate one-hot encoded features ---
ONE_HOT_GROUPS = {
    "Surgical Approach": {
        "columns": ["surgical_approach_group_1", "surgical_approach_group_2"],
        "root": "surgical_approach_group"
    },
    "Marital Status": {
        "columns": ["marital_status_1", "marital_status_2", "marital_status_3", "marital_status_4"],
        "root": "marital_status"
    },
    "Gender": {
        "columns": ["gender_1"],
        "root": "gender"
    },
    "Osteoporosis": {
        "columns": ["osteoporosis_1"],
        "root": "osteoporosis"
    },
    "Psychiatric Disorder": {
        "columns": ["psych_1"],
        "root": "psych"
    },
    "Hypertension": {
        "columns": ["htn_1"],
        "root": "htn"
    }
}

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def get_display_text(root_col, raw_value):
    """Convert numeric value to readable text based on VALUE_MAPS"""
    try:
        val_int = int(raw_value)
    except:
        return f"{raw_value:.1f}"  # Float values (continuous)
    
    # Check specific maps (gender, marital_status, etc.)
    if root_col in VALUE_MAPS:
        return VALUE_MAPS[root_col].get(val_int, str(val_int))
    
    # Check binary variables
    if root_col in BINARY_COLS:
        return VALUE_MAPS["binary"].get(val_int, str(val_int))
    
    return str(val_int)

def aggregate_shap_values(shap_values, feature_names, X_row_original, groups_cfg, pretty_names):
    """
    Aggregate SHAP values for one-hot encoded features and map to readable text.
    """
    agg_data = {}
    processed_feats = set()
    
    # 1. Process GROUPS (aggregate one-hot columns)
    for group_disp_name, cfg in groups_cfg.items():
        shap_sum = 0.0
        found = False
        
        for col_enc in cfg["columns"]:
            # Find matching feature (handle cont__ and cat__ prefixes)
            matches = [f for f in feature_names if f.endswith(col_enc)]
            if matches:
                idx = list(feature_names).index(matches[0])
                shap_sum += shap_values[idx]
                processed_feats.add(matches[0])
                found = True
        
        if found:
            # Get display value from original data
            root = cfg.get("root")
            if root and root in X_row_original:
                raw_val = X_row_original[root]
                val_text = get_display_text(root, raw_val)
                agg_data[group_disp_name] = {"shap": shap_sum, "val": val_text}
    
    # 2. Process SIMPLE features (remaining columns)
    for i, f in enumerate(feature_names):
        if f in processed_feats:
            continue
        
        # Clean feature name (remove cont__ and cat__ prefixes)
        clean = f.replace("cont__", "").replace("cat__", "")
        root_col = re.sub(r'_\d+$', '', clean)  # Remove trailing numbers
        
        # Get display name
        display_name = pretty_names.get(root_col, pretty_names.get(clean, clean))
        
        # Get display value
        val_text = "?"
        if root_col in X_row_original:
            raw_val = X_row_original[root_col]
            if isinstance(raw_val, (float, np.floating)) and root_col not in VALUE_MAPS and root_col not in BINARY_COLS:
                val_text = f"{raw_val:.1f}"
            else:
                val_text = get_display_text(root_col, raw_val)
        
        # Save (accumulate if duplicate display name)
        if display_name in agg_data:
            agg_data[display_name]["shap"] += shap_values[i]
        else:
            agg_data[display_name] = {"shap": shap_values[i], "val": val_text}
    
    return agg_data

print("✅ SHAP force plot helpers loaded")

In [ ]:
if y_true is not None:
    # Initialize SHAP
    shap.initjs()
    
    # Get expected value (bias term from SHAP)
    expected_value = shap_values[:, -1].mean()
    
    # Find correct predictions
    correct_mask = (y_true == y_pred)
    correct_idx = np.where(correct_mask)[0]
    
    if len(correct_idx) == 0:
        print("⚠ No correct predictions found - skipping force plots")
    else:
        # Separate by class
        class0_correct = correct_idx[y_true[correct_idx] == 0]  # Home
        class1_correct = correct_idx[y_true[correct_idx] == 1]  # Non-Home
        
        # Sort by confidence (distance from 0.5)
        if len(class0_correct) > 0:
            class0_conf = np.abs(y_proba[class0_correct] - 0.5)
            class0_top = class0_correct[np.argsort(class0_conf)[-CFG["N_correct_class_0"]:]]
        else:
            class0_top = []
            
        if len(class1_correct) > 0:
            class1_conf = np.abs(y_proba[class1_correct] - 0.5)
            class1_top = class1_correct[np.argsort(class1_conf)[-CFG["N_correct_class_1"]:]]
        else:
            class1_top = []
        
        print(f"✅ Selected {len(class0_top)} samples from class 0 (Home)")
        print(f"✅ Selected {len(class1_top)} samples from class 1 (Non-Home)")
        
        # Create output directory
        force_dir = Path("shap_force_plots")
        force_dir.mkdir(exist_ok=True)
        
        # Combine patients
        patients = [{'idx': i, 'lbl': 'Home', 'cls': 0} for i in class0_top] + \
                   [{'idx': i, 'lbl': 'Non-Home', 'cls': 1} for i in class1_top]
        
        # Generate force plots
        for pat in patients:
            idx = pat['idx']
            raw_sv = shap_values_clean[idx]
            row_original = df.iloc[idx]  # Original raw data
            
            # Aggregate SHAP values and map to readable text
            agg_dict = aggregate_shap_values(raw_sv, SELECTED_FEATURES, row_original, ONE_HOT_GROUPS, PRETTY_NAMES)
            
            # Sort by absolute SHAP value and select top K
            sorted_items = sorted(agg_dict.items(), key=lambda item: abs(item[1]["shap"]), reverse=True)
            top_k = sorted_items[:CFG["n_features_plot"]]
            
            plot_shap_vals = np.array([item[1]["shap"] for item in top_k])
            # Format: "Feature Name = Value"
            plot_names = [f"{item[0]} = {item[1]['val']}" for item in top_k]
            
            # Plot
            plt.figure(figsize=CFG["figsize"])
            shap.force_plot(
                expected_value,
                plot_shap_vals,
                features=None,  # Don't pass numeric values
                feature_names=plot_names,
                matplotlib=True,
                show=False,
                text_rotation=CFG["text_rotation"]
            )
            
            fname = f"force_grouped_c{pat['cls']}_idx{idx}_p{y_proba[idx]:.4f}.png"
            plt.savefig(force_dir / fname, dpi=250, bbox_inches="tight", pad_inches=0.2)
            plt.close()
            print(f"  💾 Saved: {fname} | {pat['lbl']} (Prob: {y_proba[idx]:.4f})")
        
        print(f"\n✅ All force plots saved to: {force_dir.absolute()}")
else:
    print("⚠ No ground truth - force plots skipped (requires labels to identify correct predictions)")